# GCSG Sandbox Notebook

**Companion to the 11 scored task notebooks.** This notebook is NOT on the
leaderboard — it lets you dial in any canonical GCSG task preset (T1–T11) or
a custom non-canonical variant from a single configuration cell, then runs it
against the Kaggle LLM harness.

## How it works

1. Edit the **Config cell** to either:
   - Set `canonical_task_id = "t1"` (or any of `t1..t11`) to reproduce a scored task.
   - Or set `canonical_task_id = None` and override individual parameters to
     construct a non-canonical variant.
2. Run the `@kbench.task` cell to register the task with the harness.
3. Run the `.run(kbench.llm)` cell to execute against the current Kaggle LLM.

## Canonical preset reproducibility

When `canonical_task_id` is set and all other params are default, the sandbox
reproduces the corresponding scored task within per-run variance (σ ≈ 0.05 on
T1 per the Flash N=10 pilot; §13 Decision Log 2026-04-14).

## Environment selection

By default the main study runs on ABCD (4-zone ecosystem). Set
`"environment": "EFGHI"` to run the main study entirely on the 5-zone
industrial facility instead. When EFGHI is selected:
- `split_type` must be `"custom"` (no canonical EFGHI main-study splits exist).
- Context init keys are automatically remapped (`"full-context"` →
  `"efghi-full-context"`, etc.).
- All phases use the EFGHI mission prompt builder.
- Cannot be combined with `transfer_environment="EFGHI"`.

## What's NOT supported in v1

- **Chained adaptation** (`rule A → rule B → reverse A`) — the parameter is
  accepted but raises `NotImplementedError`. Tracked as follow-on.
- **Custom adaptation rules** — use `"scarcity"` or `"integrate"` only.
- **Synthetic grammars other than ABCD/EFGHI** — not configurable here.
- **L4 on ABCD** — L4 only exists in the EFGHI dataset.

See `sandbox_notebook_spec.md` §§9, 11, 12 for the full list of non-goals and
follow-on items.

## Design invariants preserved

- **Single-session for T10/T11** (Gotcha #3, #14) — one `run_phased_session`
  per run, gate check via `gate_check` kwarg.
- **Fail-fast on gate failure** (Gotcha #4) — first gate miss → score 0.0,
  stop remaining runs.
- **Canonical splits from CSV columns** (Gotcha #5) — when `split_type="canonical"`.
- **Lenient 4-layer parser** (Gotcha #9) — inherited from gcsg-common.
- **API_FAIL excluded from score** (Gotcha #10) — inherited from gcsg-common.
- **T9 LE over EFGHI L2-L3 only** (Gotcha #12) — via `probe_level_filter=[2,3]`.

In [ ]:
# ======================================================================
# CONFIG  — edit this cell to configure the sandbox run
# ======================================================================
#
# Two modes:
#   (1) Canonical preset:  set canonical_task_id = "t1".."t11"
#   (2) Custom variant:     set canonical_task_id = None and override params
#
# See sandbox_notebook_spec.md §3 for the full parameter surface.

# ── PRESETS — 11 canonical task configurations ────────────────────────
# Each preset is a full parameter dict. The sandbox task function uses
# the preset when canonical_task_id is set, else uses the individual params.

PRESETS = {
    "t1": {
        "canonical_task_id": "t1", "environment": "ABCD",
        "split_type": "canonical", "split_option": "random-small",
        "k_option": "no_k_holdout", "prompt_style": "canonical-canonical",
        "schedule": "probe-at-end",
        "context_init_ceiling": "full-context",
        "context_init_learning": "environment-context",
        "adaptation": None, "retention": False,
        "transfer_environment": None, "probe_level_filter": None,
        "metric": "simple_le", "fail_fast": False,
    },
    "t2": {
        "canonical_task_id": "t2", "environment": "ABCD",
        "split_type": "canonical", "split_option": "random-small",
        "k_option": "no_k_holdout", "prompt_style": "canonical-canonical",
        "schedule": "probe-at-end",
        "context_init_ceiling": "full-context",
        "context_init_learning": "no-context",
        "adaptation": None, "retention": False,
        "transfer_environment": None, "probe_level_filter": None,
        "metric": "simple_le", "fail_fast": False,
    },
    "t3": {
        "canonical_task_id": "t3", "environment": "ABCD",
        "split_type": "canonical", "split_option": "random-small",
        "k_option": "no_k_holdout", "prompt_style": "canonical-canonical",
        "schedule": "interleaved-blocks", "n_blocks": 8,
        "context_init_ceiling": "full-context",
        "context_init_learning": "environment-context",
        "adaptation": None, "retention": False,
        "transfer_environment": None, "probe_level_filter": None,
        "metric": "wlc", "fail_fast": False,
    },
    "t4": {
        "canonical_task_id": "t4", "environment": "ABCD",
        "split_type": "canonical", "split_option": "goalwise-small",
        "k_option": "no_k_holdout", "prompt_style": "canonical-canonical",
        "schedule": "probe-at-end",
        "context_init_ceiling": "full-context",
        "context_init_learning": "environment-context",
        "adaptation": None, "retention": False,
        "transfer_environment": None,
        "probe_level_filter": [3],   # L3-only probe (depth extrapolation)
        "metric": "simple_le", "fail_fast": False,
    },
    "t5": {
        "canonical_task_id": "t5", "environment": "ABCD",
        "split_type": "canonical", "split_option": "holdout-small",
        "k_option": "no_k_holdout", "prompt_style": "canonical-canonical",
        "schedule": "probe-at-end",
        "context_init_ceiling": "full-context",
        "context_init_learning": "environment-context",
        "adaptation": None, "retention": False,
        "transfer_environment": None, "probe_level_filter": None,
        "metric": "simple_le", "fail_fast": False,
    },
    "t6": {
        "canonical_task_id": "t6", "environment": "ABCD",
        "split_type": "canonical", "split_option": "combo-holdout",
        "k_option": "no_k_holdout", "prompt_style": "canonical-canonical",
        "schedule": "probe-at-end",
        "context_init_ceiling": "full-context",
        "context_init_learning": "environment-context",
        "adaptation": None, "retention": False,
        "transfer_environment": None, "probe_level_filter": None,
        "metric": "simple_le", "fail_fast": False,
        "emit_per_template": True,
    },
    "t7": {
        "canonical_task_id": "t7", "environment": "ABCD",
        "split_type": "canonical", "split_option": "random-small",
        "k_option": "no_k_holdout", "prompt_style": "canonical-non_canonical",
        "schedule": "probe-at-end",
        "context_init_ceiling": "full-context",
        "context_init_learning": "environment-context",
        "adaptation": None, "retention": False,
        "transfer_environment": None, "probe_level_filter": None,
        "metric": "simple_le", "fail_fast": False,
        "emit_per_template": True,
    },
    "t8": {
        "canonical_task_id": "t8", "environment": "ABCD",
        "split_type": "canonical", "split_option": "random-small",
        "k_option": "drastic_k_holdout", "prompt_style": "canonical-canonical",
        "schedule": "probe-at-end",
        "context_init_ceiling": "full-context",
        "context_init_learning": "environment-context",
        "adaptation": None, "retention": False,
        "transfer_environment": None, "probe_level_filter": None,
        "metric": "simple_le", "fail_fast": False,
    },
    "t9": {
        "canonical_task_id": "t9", "environment": "ABCD",
        "split_type": "canonical", "split_option": "random-small",
        "k_option": "no_k_holdout", "prompt_style": "canonical-canonical",
        "schedule": "probe-at-end",
        "context_init_ceiling": "full-context",
        "context_init_learning": "environment-context",
        "adaptation": None, "retention": False,
        "transfer_environment": "EFGHI", "include_verify_probe": True,
        "probe_level_filter": [2, 3],   # T9 LE over EFGHI L2-L3 only (Gotcha #12)
        "metric": "simple_le", "fail_fast": False,
        "emit_per_template": True,
    },
    "t10": {
        "canonical_task_id": "t10", "environment": "ABCD",
        "split_type": "canonical", "split_option": "random-large",
        "k_option": "no_k_holdout", "prompt_style": "canonical-canonical",
        "schedule": "probe-at-end",
        "context_init_learning": "environment-context",
        "adaptation": "scarcity", "retention": False,
        "transfer_environment": None, "probe_level_filter": None,
        "metric": "plasticity",
        "fail_fast": True, "learning_gate_threshold": 0.40,
    },
    "t11": {
        "canonical_task_id": "t11", "environment": "ABCD",
        "split_type": "canonical", "split_option": "random-large",
        "k_option": "no_k_holdout", "prompt_style": "canonical-canonical",
        "schedule": "probe-at-end",
        "context_init_learning": "environment-context",
        "adaptation": "integrate", "retention": True,
        "transfer_environment": None, "probe_level_filter": None,
        "metric": "stability",
        "fail_fast": True, "learning_gate_threshold": 0.40,
    },
}

# ── ACTIVE CONFIG ─────────────────────────────────────────────────────
# Option A: reproduce a scored task. Set canonical_task_id and move on.
# Option B: pass a custom dict. Set canonical_task_id = None and override params.

SANDBOX_CONFIG = PRESETS["t1"]       # ← default: reproduce T1 on Flash

# ── MULTI-RUN CONTROLS (apply in both modes) ──────────────────────────
SANDBOX_CONFIG = {**SANDBOX_CONFIG,
    "n_runs": 3,           # independent runs with different presentation orders
    "base_seed": 1,        # per-run seed = base_seed + run_idx
    "emit_secondaries": True,
}

# ======================================================================
# RECIPES  — copy-paste one of these over SANDBOX_CONFIG for a variant
# ======================================================================
#
# # Recipe 1: T4 productivity but also restricted to the T5 holdouts.
# # (Not a scored task — a combined-axis probe of depth + configuration.)
# SANDBOX_CONFIG = {
#     "canonical_task_id": None,
#     "split_type": "custom", "split_option": "holdout-small",
#     "k_option": "no_k_holdout", "prompt_style": "canonical-canonical",
#     "schedule": "probe-at-end",
#     "context_init_ceiling": "full-context",
#     "context_init_learning": "environment-context",
#     "adaptation": None, "retention": False,
#     "transfer_environment": None, "probe_level_filter": [3],
#     "metric": "simple_le", "fail_fast": False,
#     "n_runs": 3, "base_seed": 1,
# }
#
# # Recipe 2: T8 but with even more extreme k-severity extrapolation.
# # (Requires k options from K_SEVERITY_OPTIONS — "drastic_k_holdout" has
# # the largest gap currently shipped; a custom dead zone requires a package patch.)
# SANDBOX_CONFIG = {
#     "canonical_task_id": None,
#     "split_type": "custom", "split_option": "random-small",
#     "k_option": "drastic_k_holdout", "prompt_style": "canonical-canonical",
#     "schedule": "probe-at-end",
#     "context_init_ceiling": "full-context",
#     "context_init_learning": "environment-context",
#     "adaptation": None, "retention": False,
#     "transfer_environment": None, "probe_level_filter": None,
#     "metric": "simple_le", "fail_fast": False,
#     "n_runs": 3, "base_seed": 1,
# }
#
# # Recipe 3: T9 transfer but INCLUDING L4 in the primary LE numerator.
# # (T9 canonical reports L4 only as secondary; this treats L4 as primary.)
# SANDBOX_CONFIG = {
#     "canonical_task_id": "t9",
#     "probe_level_filter": [2, 3, 4],
#     "n_runs": 3, "base_seed": 1,
# }
#
# # Recipe 4: T1 but with the cheap pilot N=1 budget.
# SANDBOX_CONFIG = {**PRESETS["t1"], "n_runs": 1, "base_seed": 1}
#
# # Recipe 5: T10 plasticity with a stricter learning gate (0.60 instead of 0.40).
# # Fail-fast on the first gate miss; useful for narrow-band model screening.
# SANDBOX_CONFIG = {**PRESETS["t10"], "learning_gate_threshold": 0.60,
#                   "n_runs": 3, "base_seed": 1}
#
# # Recipe 6: Main study on EFGHI (5-zone industrial facility).
# # Runs a T1-equivalent (rule induction) entirely on EFGHI — no transfer,
# # no ABCD at all. Context init is remapped to EFGHI's own baseline/environment.
# # Requires split_type="custom" (no canonical EFGHI main-study splits exist).
# # EFGHI has L1=10, L2=20, L3=40, L4=80 rows, so random-small (24/24)
# # should be feasible if the L1 pool has enough variant diversity.
# SANDBOX_CONFIG = {
#     "canonical_task_id": None,
#     "environment": "EFGHI",
#     "split_type": "custom", "split_option": "random-small",
#     "k_option": "no_k_holdout", "prompt_style": "canonical-canonical",
#     "schedule": "probe-at-end",
#     "context_init_ceiling": "full-context",
#     "context_init_learning": "environment-context",
#     "adaptation": None, "retention": False,
#     "transfer_environment": None, "probe_level_filter": None,
#     "metric": "simple_le", "fail_fast": False,
#     "n_runs": 3, "base_seed": 1,
# }


In [ ]:
# ======================================================================
# @kbench.task — SELF-CONTAINED for Kaggle Benchmarks eval
# ======================================================================
# Reads SANDBOX_CONFIG from module globals (set in the Config cell above).
# The decorator fires once, at notebook load time; the function body runs
# once per call. Because the Kaggle harness re-executes every cell before
# calling the task, SANDBOX_CONFIG is always present when this runs.

@kbench.task(name="gcsg_sandbox")
def gcsg_sandbox(llm) -> float:
    """GCSG Sandbox — reproduce any canonical task or a custom variant.

    See the config cell for the full parameter surface.
    """
    # ── 1. Install dependencies ──────────────────────────────────────
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "https://github.com/haesoo-park/gcsg-benchmark/archive/refs/tags/v0.8.6.tar.gz",
        "-q", "--disable-pip-version-check",
    ])

    # ── 2. Imports ───────────────────────────────────────────────────
    import json, io, math, pandas as pd
    import importlib.resources as _res
    import kaggle_benchmarks as kbench
    from common.core import compile_environment_runtime, RuleModifiers
    from common.session import (
        APIFailError, GateFailedError, PhaseSpec, RULE_PROMPTS,
        RULE_TYPE_TO_MODIFIERS, run_phased_session,
    )
    from common.metrics import (
        compute_learning_efficiency, compute_plasticity, compute_stability,
    )
    from common.task_splits import (
        allocate_split, build_adaptation_pools, order_missions_for_run,
        K_SEVERITY_OPTIONS, K_SEVERITY_OPTIONS_ADAPTATION,
    )
    from common.prompts import (
        build_efghi_mission_text, build_transfer_transition_prompt,
    )

    # ── 3. ABCD data ────────────────────────────────────────────────
    _data = _res.files("common.data")
    task_config = json.loads((_data / "task_matrix.json").read_text())["tasks"][0]
    env_runtime = compile_environment_runtime(
        json.loads((_data / "environment_spec.json").read_text())
    )
    task_df = pd.read_csv(io.StringIO(
        (_data / "dataset_state_complete.csv").read_text()
    ))

    # ── 3b. EFGHI data (loaded eagerly; unused when transfer_environment=None) ─
    efghi_config = json.loads((_data / "efghi_task_matrix.json").read_text())["tasks"][0]
    efghi_runtime = compile_environment_runtime(
        json.loads((_data / "efghi_environment_spec.json").read_text())
    )
    efghi_df = pd.read_csv(io.StringIO(
        (_data / "efghi_dataset_sequence_complete.csv").read_text()
    ))

    # ── 3c. Score finalizer ─────────────────────────────────────────
    def _finalize_score(raw_score):
        if raw_score is None:
            return float("nan")
        value = float(raw_score)
        if math.isnan(value):
            return float("nan")
        return max(0.0, min(1.0, value))

    # ── 3d. Secondaries emitter (pooled across runs) ────────────────
    def _emit_secondaries(all_run_results, task_id, by_template=False):
        from collections import defaultdict
        _pools  = defaultdict(lambda: [0, 0])
        _fvs    = defaultdict(lambda: [0, 0])
        _tmpls  = defaultdict(lambda: [0, 0])
        _styles = defaultdict(lambda: [0, 0])
        for _run in all_run_results:
            for _cond, _res in (_run.get("results") or {}).items():
                if _res is None:
                    continue
                _tr = _res.get("all_traces")
                if _tr is None or getattr(_tr, "empty", True):
                    continue
                for _, _row in _tr.iterrows():
                    _phase = str(_row.get("phase_name", ""))
                    _lvl   = _row.get("goal_level")
                    if not _phase or pd.isna(_lvl):
                        continue
                    _lvl = int(_lvl)
                    _ok  = int(bool(_row.get("mission_success", 0)))
                    _pools[(_cond, _phase, _lvl)][0] += _ok
                    _pools[(_cond, _phase, _lvl)][1] += 1
                    _flbl = str(_row.get("failure_label", "NONE"))
                    _fvs[(_cond, _phase)][0] += int(_flbl != "FORMAT_PARSE_FAIL")
                    _fvs[(_cond, _phase)][1] += 1
                    if by_template and "probe" in _phase.lower():
                        _tid = str(_row.get("template_id", ""))
                        if _tid:
                            _tmpls[(_cond, _phase, _tid)][0] += _ok
                            _tmpls[(_cond, _phase, _tid)][1] += 1
                        _sty = str(_row.get("prompt_style", ""))
                        if _sty:
                            _styles[(_cond, _phase, _sty)][0] += _ok
                            _styles[(_cond, _phase, _sty)][1] += 1
        for (_cond, _phase, _lvl), (_k, _n) in sorted(_pools.items()):
            _fk, _fn = _fvs.get((_cond, _phase), [0, 0])
            print("SECONDARY_JSON " + json.dumps({
                "task": task_id, "n_runs": len(all_run_results),
                "condition": _cond, "phase": _phase, "level": _lvl,
                "k": _k, "n": _n,
                "acc": round(_k / _n, 4) if _n else None,
                "format_valid_rate": round(_fk / _fn, 4) if _fn else None,
            }))
        for (_cond, _phase, _tid), (_k, _n) in sorted(_tmpls.items()):
            print("SECONDARY_JSON " + json.dumps({
                "task": task_id, "n_runs": len(all_run_results),
                "condition": _cond, "phase": _phase, "template_id": _tid,
                "k": _k, "n": _n,
                "acc": round(_k / _n, 4) if _n else None,
            }))
        if len({s for (_c, _p, s) in _styles}) > 1:
            for (_cond, _phase, _sty), (_k, _n) in sorted(_styles.items()):
                print("SECONDARY_JSON " + json.dumps({
                    "task": task_id, "n_runs": len(all_run_results),
                    "condition": _cond, "phase": _phase, "prompt_style": _sty,
                    "k": _k, "n": _n,
                    "acc": round(_k / _n, 4) if _n else None,
                }))
        if _pools:
            print("\n─── SECONDARY METRICS (pooled across runs) ───")
            _by = {}
            for (_cond, _phase, _lvl), (_k, _n) in sorted(_pools.items()):
                _by.setdefault(_cond, {}).setdefault(_phase, []).append((_lvl, _k, _n))
            for _cond in sorted(_by):
                for _phase in sorted(_by[_cond]):
                    _cells = "  ".join(f"L{_l}:{_k}/{_n}={_k/_n:.3f}"
                                       for _l, _k, _n in sorted(_by[_cond][_phase]))
                    _fk, _fn = _fvs.get((_cond, _phase), [0, 0])
                    _fv = f"  fmt_valid={_fk/_fn:.3f}" if _fn else ""
                    print(f"  [{_cond}] {_phase:18s} {_cells}{_fv}")
            if len({s for (_c, _p, s) in _styles}) > 1:
                print("  per prompt_style:")
                _sby = {}
                for (_cond, _phase, _sty), (_k, _n) in sorted(_styles.items()):
                    _sby.setdefault((_cond, _phase), []).append((_sty, _k, _n))
                for (_cond, _phase), _rows in sorted(_sby.items()):
                    _cells = "  ".join(f"{_s}:{_k}/{_n}={_k/_n:.3f}"
                                       for _s, _k, _n in sorted(_rows))
                    print(f"    [{_cond}] {_phase:15s} {_cells}")

    import numpy as np


    # ── 4. Resolve config ────────────────────────────────────────────
    try:
        cfg = dict(globals()["SANDBOX_CONFIG"])
    except KeyError:
        raise RuntimeError(
            "SANDBOX_CONFIG not found in globals. The config cell must define it "
            "(e.g. SANDBOX_CONFIG = PRESETS['t1']) before running the task."
        )

    # Canonical preset override
    preset_id = cfg.get("canonical_task_id")
    if preset_id is not None:
        presets = globals().get("PRESETS", {})
        if preset_id not in presets:
            raise ValueError(
                f"canonical_task_id='{preset_id}' not in PRESETS. "
                f"Valid: {sorted(presets.keys())}"
            )
        # Preset wins over individual parameters when set
        merged = dict(presets[preset_id])
        # Carry through things that don't come from the preset (if user set them)
        for k in ("n_runs", "base_seed", "emit_secondaries", "emit_per_template"):
            if k in cfg:
                merged[k] = cfg[k]
        cfg = merged

    # Defaults
    cfg.setdefault("n_runs", 3)
    cfg.setdefault("base_seed", 1)
    cfg.setdefault("emit_secondaries", True)
    cfg.setdefault("split_type", "canonical")
    cfg.setdefault("split_option", "random-small")
    cfg.setdefault("k_option", "no_k_holdout")
    cfg.setdefault("prompt_style", "canonical-canonical")
    cfg.setdefault("schedule", "probe-at-end")
    cfg.setdefault("n_blocks", 8)
    cfg.setdefault("context_init_ceiling", "full-context")
    cfg.setdefault("context_init_learning", "environment-context")
    cfg.setdefault("adaptation", None)
    cfg.setdefault("retention", False)
    cfg.setdefault("transfer_environment", None)
    cfg.setdefault("include_verify_probe", True)
    cfg.setdefault("probe_level_filter", None)
    cfg.setdefault("metric", "simple_le")
    cfg.setdefault("fail_fast", cfg.get("adaptation") is not None)
    cfg.setdefault("learning_gate_threshold", 0.40)
    cfg.setdefault("comprehension_threshold", 0.50)
    cfg.setdefault("chained_adaptation", None)
    cfg.setdefault("environment", "ABCD")
    cfg.setdefault("adaptation_signal", "change-only")
    cfg.setdefault("reflection_after_feedback", False)

    # Unsupported-in-v1 guards
    if cfg.get("chained_adaptation") is not None:
        raise NotImplementedError(
            "chained_adaptation is not implemented in sandbox v1 — see "
            "sandbox_notebook_spec.md §3.7 and §9."
        )
    if cfg.get("adaptation") == "custom":
        raise NotImplementedError(
            "adaptation='custom' is not implemented in sandbox v1."
        )
    if cfg.get("reflection_after_feedback"):
        print("WARNING: reflection_after_feedback=True is a legacy knob; see "
              "agent_context_prompt.md Gotcha #19. Proceeding anyway.")

    N_RUNS = int(cfg["n_runs"])
    BASE_SEED = int(cfg["base_seed"])
    METRIC = str(cfg["metric"])
    K_OPT = str(cfg["k_option"])
    PROMPT_STYLE = str(cfg["prompt_style"])
    SPLIT_OPTION = str(cfg["split_option"])
    SCHEDULE = str(cfg["schedule"])
    N_BLOCKS = int(cfg["n_blocks"])
    ADAPTATION = cfg["adaptation"]
    RETENTION = bool(cfg["retention"])
    TRANSFER_ENV = cfg["transfer_environment"]
    INCLUDE_VERIFY = bool(cfg["include_verify_probe"])
    PROBE_LEVEL_FILTER = cfg["probe_level_filter"]
    CTX_CEIL = str(cfg["context_init_ceiling"])
    CTX_LEARN = str(cfg["context_init_learning"])
    FAIL_FAST = bool(cfg["fail_fast"])
    GATE = float(cfg["learning_gate_threshold"])
    ENVIRONMENT = str(cfg.get("environment", "ABCD"))
    TASK_ID = f"sandbox-{preset_id}" if preset_id else "sandbox-custom"

    # ── 4b. Environment swap ─────────────────────────────────────────
    # When environment="EFGHI", the main study runs entirely on the EFGHI
    # dataset (5-zone industrial facility) instead of ABCD (4-zone ecosystem).
    # This replaces task_df / env_runtime / task_config and remaps context_init
    # to the EFGHI-specific variants already supported by common.prompts.
    _EFGHI_CTX_MAP = {
        "full-context": "efghi-full-context",
        "environment-context": "efghi-environment-context",
    }
    _efghi_mission_builder = None  # set below only when EFGHI is the main env

    if ENVIRONMENT == "EFGHI":
        # Swap datasets so the main study operates on EFGHI
        task_df = efghi_df
        env_runtime = efghi_runtime
        task_config = efghi_config
        _efghi_mission_builder = lambda row, style: build_efghi_mission_text(row=row)

        # Remap context-init keys to EFGHI variants
        CTX_CEIL = _EFGHI_CTX_MAP.get(CTX_CEIL, CTX_CEIL)
        CTX_LEARN = _EFGHI_CTX_MAP.get(CTX_LEARN, CTX_LEARN)

        # Validation: EFGHI main + EFGHI transfer is nonsensical
        if TRANSFER_ENV == "EFGHI":
            raise ValueError(
                "environment='EFGHI' with transfer_environment='EFGHI' is "
                "not meaningful — there is no environment to transfer FROM. "
                "Use transfer_environment=None for a pure EFGHI main study, "
                "or environment='ABCD' with transfer_environment='EFGHI' for "
                "the T9-style cross-environment transfer."
            )

        # Canonical splits do not exist for EFGHI main-study tasks
        if cfg["split_type"] == "canonical":
            raise ValueError(
                "split_type='canonical' is not available when environment='EFGHI'. "
                "No canonical tN_split columns exist for EFGHI main-study tasks "
                "(only t9_split for the transfer-probe split). "
                "Use split_type='custom' with a compatible split_option."
            )

        print(f"ENVIRONMENT = EFGHI (5 zones) — main study dataset swapped")
        print(f"  context_init remapped: ceiling={CTX_CEIL!r}, learning={CTX_LEARN!r}")
    elif ENVIRONMENT != "ABCD":
        raise ValueError(f"environment must be 'ABCD' or 'EFGHI', got {ENVIRONMENT!r}")

    # emit_per_template auto-enables for combo-holdout, non-canonical style, and transfer
    emit_per_template = bool(cfg.get("emit_per_template")
        or SPLIT_OPTION == "combo-holdout"
        or PROMPT_STYLE == "canonical-non_canonical"
        or TRANSFER_ENV is not None
        or ENVIRONMENT == "EFGHI")

    print("─── Sandbox config (resolved) ───")
    for _k in ("canonical_task_id", "environment", "split_type", "split_option",
               "k_option", "prompt_style", "schedule", "adaptation", "retention",
               "transfer_environment", "metric", "n_runs", "fail_fast"):
        print(f"  {_k:28s} = {cfg.get(_k)!r}")

    # ── 5. Build main-study split ────────────────────────────────────
    # task_df / env_runtime already point to the correct dataset per §4b.
    if cfg["split_type"] == "canonical":
        if preset_id is None:
            raise ValueError(
                "split_type='canonical' requires canonical_task_id to be set. "
                "For custom splits use split_type='custom'."
            )
        task_number = int(preset_id.lstrip("t"))
        split = allocate_split(
            task_df, env_runtime, split_option=SPLIT_OPTION,
            use_canonical=True, task_number=task_number,
        )
    else:
        split = allocate_split(
            task_df, env_runtime, split_option=SPLIT_OPTION, seed=BASE_SEED,
        )
    train_rows = split.train_rows
    probe_rows = split.probe_rows
    print(f"{ENVIRONMENT} split: train={split.summary['train_total']} {split.summary['train_by_level']}  "
          f"probe={split.summary['probe_total']} {split.summary['probe_by_level']}")

    # ── 5b. Build transfer (EFGHI) split if requested ────────────────
    efghi_probe_rows = []
    if TRANSFER_ENV == "EFGHI":
        if cfg["split_type"] == "canonical" and preset_id == "t9":
            efghi_split = allocate_split(
                efghi_df, efghi_runtime, split_option="transfer-probe",
                use_canonical=True, task_number=9,
            )
        else:
            efghi_split = allocate_split(
                efghi_df, efghi_runtime, split_option="transfer-probe",
                seed=BASE_SEED,
            )
        efghi_probe_rows = efghi_split.probe_rows
        print(f"EFGHI split: probe={len(efghi_probe_rows)}")

    # ── 6. k-severity resolution ─────────────────────────────────────
    if K_OPT not in K_SEVERITY_OPTIONS:
        raise ValueError(f"k_option='{K_OPT}' not in {sorted(K_SEVERITY_OPTIONS.keys())}")
    _bq_train = K_SEVERITY_OPTIONS[K_OPT]["train"]
    _bq_test  = K_SEVERITY_OPTIONS[K_OPT]["test"]
    if ADAPTATION is not None:
        _bq_at = K_SEVERITY_OPTIONS_ADAPTATION[K_OPT]["train"]
        _bq_ap = K_SEVERITY_OPTIONS_ADAPTATION[K_OPT]["test"]
    _bq_efghi = None
    if TRANSFER_ENV == "EFGHI":
        _bq_efghi = {
            "min": efghi_config["bad_quality_level_range"]["min"],
            "max": efghi_config["bad_quality_level_range"]["max"],
        }

    # ── 7. Experiment conditions ─────────────────────────────────────
    # Non-adaptation paths use ceiling + learning (except T2 uses no-context).
    # Adaptation paths (T10/T11) use a single learning condition.
    if ADAPTATION is not None:
        EXPERIMENT_CONDITIONS = [
            {
                "label": "learning",
                "context_init": CTX_LEARN,
                "main_feedback": "feedback_only",
                "adapt_injection": RULE_PROMPTS[ADAPTATION]["injection_baseline"],
                "adapt_feedback": "feedback_only",
                "ret_reversal": RULE_PROMPTS[ADAPTATION].get("reversal_baseline", ""),
            },
        ]
    elif TRANSFER_ENV == "EFGHI":
        EXPERIMENT_CONDITIONS = [
            {"label": "ceiling",  "context_init": "full-context",
             "feedback_mode": "none", "probe_only": True,
             "transfer_prompt": build_transfer_transition_prompt("ceiling")},
            {"label": "learning", "context_init": CTX_LEARN,
             "feedback_mode": "feedback_only", "probe_only": False,
             "transfer_prompt": build_transfer_transition_prompt("learning")},
        ]
    else:
        EXPERIMENT_CONDITIONS = [
            {"label": "ceiling",  "context_init": CTX_CEIL,
             "feedback_mode": "none", "probe_only": True},
            {"label": "learning", "context_init": CTX_LEARN,
             "feedback_mode": "feedback_only", "probe_only": False},
        ]

    # ── 8. Phase builders ────────────────────────────────────────────
    def _assign_blocks(rows, n_blocks):
        """Level-balanced block assignment (T3 interleaved schedule)."""
        by_level = {}
        for r in rows:
            lv = r["_goal_level"]
            by_level.setdefault(lv, []).append(r)
        blocks = [[] for _ in range(n_blocks)]
        for lv in sorted(by_level):
            for i, r in enumerate(by_level[lv]):
                blocks[i % n_blocks].append(r)
        return blocks

    def _build_simple_phases(cond, ordered_train, ordered_probe):
        """train → probe (optionally probe-only for ceiling)."""
        _fb = cond["feedback_mode"]
        _probe_only = cond.get("probe_only", False)
        phases = []
        if not _probe_only:
            phases.append(PhaseSpec(
                name="main_train", stage_label="train",
                mission_rows=list(ordered_train), inject_feedback=True,
                feedback_mode=_fb, bad_quality_range=_bq_train,
                mission_text_builder=_efghi_mission_builder,
            ))
        phases.append(PhaseSpec(
            name="main_probe", stage_label="probe",
            mission_rows=list(ordered_probe), inject_feedback=False,
            feedback_mode="none", bad_quality_range=_bq_test,
            mission_text_builder=_efghi_mission_builder,
        ))
        return phases

    def _build_interleaved_phases(cond, ordered_train, ordered_probe):
        """T3-style: N interleaved blocks of (train → probe)."""
        _fb = cond["feedback_mode"]
        _probe_only = cond.get("probe_only", False)
        t_blocks = _assign_blocks(ordered_train, N_BLOCKS)
        p_blocks = _assign_blocks(ordered_probe, N_BLOCKS)
        phases = []
        if _probe_only:
            for i in range(N_BLOCKS):
                phases.append(PhaseSpec(
                    name=f"probe_block_{i+1}", stage_label="probe",
                    mission_rows=list(p_blocks[i]), inject_feedback=False,
                    feedback_mode="none", bad_quality_range=_bq_test,
                    mission_text_builder=_efghi_mission_builder,
                ))
        else:
            for i in range(N_BLOCKS):
                phases.append(PhaseSpec(
                    name=f"train_block_{i+1}", stage_label="train",
                    mission_rows=list(t_blocks[i]), inject_feedback=True,
                    feedback_mode=_fb, bad_quality_range=_bq_train,
                    mission_text_builder=_efghi_mission_builder,
                ))
                phases.append(PhaseSpec(
                    name=f"probe_block_{i+1}", stage_label="probe",
                    mission_rows=list(p_blocks[i]), inject_feedback=False,
                    feedback_mode="none", bad_quality_range=_bq_test,
                    mission_text_builder=_efghi_mission_builder,
                ))
        return phases

    def _build_transfer_phases(cond, ordered_train, ordered_verify, ordered_efghi):
        """T9-style: (ABCD train + verify) → transfer prompt → EFGHI probe."""
        _fb = cond["feedback_mode"]
        _probe_only = cond.get("probe_only", False)
        phases = []
        if not _probe_only:
            phases.append(PhaseSpec(
                name="abcd_train", stage_label="train",
                mission_rows=list(ordered_train), inject_feedback=True,
                feedback_mode=_fb, bad_quality_range=_bq_train,
            ))
            if INCLUDE_VERIFY and ordered_verify:
                phases.append(PhaseSpec(
                    name="abcd_verify", stage_label="verify",
                    mission_rows=list(ordered_verify), inject_feedback=False,
                    feedback_mode="none", bad_quality_range=_bq_train,
                ))
        phases.append(PhaseSpec(
            name="transfer", stage_label="transfer",
            mission_rows=[], inject_feedback=False,
            pre_phase_prompt=cond["transfer_prompt"],
        ))
        phases.append(PhaseSpec(
            name="efghi_probe", stage_label="probe",
            mission_rows=list(ordered_efghi), inject_feedback=False,
            feedback_mode="none", bad_quality_range=_bq_efghi,
            env_runtime_override=efghi_runtime,
            mission_text_builder=lambda row, style: build_efghi_mission_text(row=row),
        ))
        return phases

    def _build_adaptation_phases(cond, ordered_train, ordered_probe, adapt_pools, rule_mods):
        """T10/T11-style: main → rule injection → interleaved adapt blocks → [reversal → retention]."""
        phases = [
            PhaseSpec(
                name="main_train", stage_label="train",
                mission_rows=list(ordered_train), inject_feedback=True,
                feedback_mode=cond["main_feedback"], bad_quality_range=_bq_train,
                mission_text_builder=_efghi_mission_builder,
            ),
            PhaseSpec(
                name="main_probe", stage_label="probe",
                mission_rows=list(ordered_probe), inject_feedback=False,
                feedback_mode="none", bad_quality_range=_bq_test,
                mission_text_builder=_efghi_mission_builder,
            ),
            PhaseSpec(
                name="rule_injection", stage_label="rule_injection",
                mission_rows=[], inject_feedback=False,
                pre_phase_prompt=cond["adapt_injection"],
            ),
        ]
        for i, (tb, pb) in enumerate(zip(adapt_pools.train_blocks, adapt_pools.probe_blocks)):
            phases.append(PhaseSpec(
                name=f"adapt_train_{i+1}", stage_label="adapt_train",
                mission_rows=list(tb), inject_feedback=True, rules=rule_mods,
                feedback_mode=cond["adapt_feedback"], bad_quality_range=_bq_at,
                mission_text_builder=_efghi_mission_builder,
            ))
            phases.append(PhaseSpec(
                name=f"adapt_probe_{i+1}", stage_label="adapt_probe",
                mission_rows=list(pb), inject_feedback=False, rules=rule_mods,
                feedback_mode="none", bad_quality_range=_bq_ap,
                mission_text_builder=_efghi_mission_builder,
            ))
        if len(adapt_pools.train_blocks) > len(adapt_pools.probe_blocks):
            last = len(adapt_pools.probe_blocks)
            phases.append(PhaseSpec(
                name=f"adapt_train_{last+1}", stage_label="adapt_train",
                mission_rows=list(adapt_pools.train_blocks[last]),
                inject_feedback=True, rules=rule_mods,
                feedback_mode=cond["adapt_feedback"], bad_quality_range=_bq_at,
                mission_text_builder=_efghi_mission_builder,
            ))
        if adapt_pools.final_probe_rows:
            phases.append(PhaseSpec(
                name="adapt_probe_final", stage_label="adapt_probe",
                mission_rows=list(adapt_pools.final_probe_rows),
                inject_feedback=False, rules=rule_mods,
                feedback_mode="none", bad_quality_range=_bq_ap,
                mission_text_builder=_efghi_mission_builder,
            ))
        if RETENTION:
            phases.append(PhaseSpec(
                name="rule_reversal", stage_label="rule_reversal",
                mission_rows=[], inject_feedback=False,
                pre_phase_prompt=cond["ret_reversal"],
            ))
            phases.append(PhaseSpec(
                name="retention_probe", stage_label="retention",
                mission_rows=list(ordered_probe), inject_feedback=False,
                feedback_mode="none", bad_quality_range=_bq_test,
                mission_text_builder=_efghi_mission_builder,
            ))
        return phases

    # ── 9. Build verify rows for transfer (level-balanced 3xL1 + 3xL2 + 2xL3) ─
    abcd_verify_rows = []
    if TRANSFER_ENV == "EFGHI" and INCLUDE_VERIFY:
        _by_lv = {}
        for _r in split.probe_rows:
            _by_lv.setdefault(int(_r["goal_level"]), []).append(_r)
        abcd_verify_rows = (
            _by_lv.get(1, [])[:3] + _by_lv.get(2, [])[:3] + _by_lv.get(3, [])[:2]
        )

    # ── 10. Adaptation pools (built once; reused per run) ────────────
    adapt_pools = None
    rule_mods = None
    if ADAPTATION is not None:
        rule_mods = RULE_TYPE_TO_MODIFIERS[ADAPTATION]
        adapt_pools = build_adaptation_pools(
            train_rows, probe_rows, env_runtime,
            rule_type=ADAPTATION, seed=BASE_SEED,
        )
        print(f"Adaptation pools ({ADAPTATION}): {adapt_pools.summary}")

    # ── 11. Run helper ───────────────────────────────────────────────
    def _run_condition(cond, phases, session_label, gate_check=None):
        return run_phased_session(
            llm=llm, kbench=kbench,
            phases=phases,
            task_config=task_config, env_runtime=env_runtime,
            context_init_option=cond["context_init"],
            prompt_style_config=PROMPT_STYLE,
            enable_live_mission_log=True,
            session_label=session_label,
            gate_check=gate_check,
        )

    def _phase_acc(result, phase):
        if result is None:
            return None
        pm = result["phase_metrics"].get(phase)
        if pm is None:
            return None
        return float(pm.get("final_probe_accuracy", pm.get("final_train_accuracy", 0)))

    def _probe_acc_filtered(result, phase, levels):
        """Accuracy over probe rows restricted to a level subset."""
        if result is None:
            return None
        traces = result.get("all_traces")
        if traces is None or traces.empty:
            return None
        sub = traces[traces["phase_name"] == phase]
        if levels is not None:
            sub = sub[sub["goal_level"].isin(list(levels))]
        return float(sub["mission_success"].mean()) if not sub.empty else None

    # ── 12. Multi-run loop ───────────────────────────────────────────
    api_fail_reason = None
    all_run_results = []
    run_scores = []
    run_details = []
    gate_failed_early = False

    # WLC helpers (T3-like metric)
    _WLC_WEIGHTS = [9 - i for i in range(1, N_BLOCKS + 1)]
    _WLC_SUM = sum(_WLC_WEIGHTS)

    def _wlc(result):
        if result is None:
            return None
        pm = result["phase_metrics"]
        total = 0.0
        for i in range(N_BLOCKS):
            block = pm.get(f"probe_block_{i+1}")
            if block is None:
                return None
            total += _WLC_WEIGHTS[i] * float(block.get("final_probe_accuracy", 0))
        return total / _WLC_SUM

    try:
        for run_idx in range(N_RUNS):
            ordered_train = order_missions_for_run(train_rows, run_idx)
            ordered_probe = order_missions_for_run(probe_rows, run_idx)
            ordered_verify = (order_missions_for_run(abcd_verify_rows, run_idx)
                              if abcd_verify_rows else [])
            ordered_efghi = (order_missions_for_run(efghi_probe_rows, run_idx)
                             if efghi_probe_rows else [])

            # ── Adaptation path (T10/T11) ──────────────────────────────
            if ADAPTATION is not None:
                cond = EXPERIMENT_CONDITIONS[0]
                full_phases = _build_adaptation_phases(
                    cond, ordered_train, ordered_probe, adapt_pools, rule_mods,
                )
                gate_spec = {
                    "after_phase": "main_probe",
                    "metric": "final_probe_accuracy",
                    "threshold": GATE,
                } if FAIL_FAST else None
                full_result = None
                probe_acc = None
                try:
                    full_result = _run_condition(cond, full_phases,
                                                 session_label=f"learning_run{run_idx}",
                                                 gate_check=gate_spec)
                    probe_acc = _phase_acc(full_result, "main_probe")
                    all_run_results.append({"run": run_idx,
                                            "results": {"learning": full_result}})
                except GateFailedError as gf:
                    probe_acc = gf.observed
                    if gf.partial_result:
                        all_run_results.append({"run": run_idx,
                                                "results": {"learning": gf.partial_result}})
                    run_details.append({
                        "run": run_idx,
                        "main_probe_acc": round(probe_acc, 4),
                        "gate_pass": False,
                        "score": None,
                    })
                    print(f"Run {run_idx+1}/{N_RUNS}: probe_acc={probe_acc:.3f} — GATE FAILED, fail-fast")
                    gate_failed_early = True
                    break

                gate_pass = probe_acc is not None and probe_acc >= GATE
                info = {
                    "run": run_idx,
                    "main_probe_acc": round(probe_acc, 4) if probe_acc is not None else None,
                    "gate_pass": gate_pass,
                }

                if METRIC == "plasticity":
                    aff_rows, naff_rows = [], []
                    for pn, pdf in full_result.get("phase_traces", {}).items():
                        if not pn.startswith("adapt_probe"):
                            continue
                        if "_scarcity_affected" not in pdf.columns:
                            continue
                        aff_rows.append(pdf[pdf["_scarcity_affected"] == True])
                        naff_rows.append(pdf[pdf["_scarcity_affected"] == False])
                    aff_df  = pd.concat(aff_rows,  ignore_index=True) if aff_rows  else pd.DataFrame()
                    naff_df = pd.concat(naff_rows, ignore_index=True) if naff_rows else pd.DataFrame()
                    aff_acc  = float(aff_df["mission_success"].mean())  if len(aff_df)  else None
                    naff_acc = float(naff_df["mission_success"].mean()) if len(naff_df) else None
                    info["affected_adapt_probe_acc"] = aff_acc
                    info["not_affected_adapt_probe_acc"] = naff_acc
                    if aff_acc is not None and naff_acc is not None and naff_acc > 0:
                        pd_out = compute_plasticity(affected_accuracy=aff_acc,
                                                    not_affected_accuracy=naff_acc)
                        score = float(pd_out["plasticity"])
                    else:
                        score = None
                elif METRIC == "stability":
                    ret_acc  = _phase_acc(full_result, "retention_probe")
                    main_acc = _phase_acc(full_result, "main_probe")
                    info["retention_probe_acc"] = ret_acc
                    if ret_acc is not None and main_acc is not None and main_acc > 0:
                        sd_out = compute_stability(retention_accuracy=ret_acc,
                                                   main_study_accuracy=main_acc)
                        score = float(sd_out["stability"])
                    else:
                        score = None
                else:
                    raise ValueError(
                        f"metric='{METRIC}' not valid for adaptation path "
                        f"(expected 'plasticity' or 'stability')"
                    )
                info["score"] = round(score, 4) if score is not None else None
                run_details.append(info)
                if score is not None:
                    run_scores.append(score)
                print(f"Run {run_idx+1}/{N_RUNS}: probe_acc={probe_acc:.3f} "
                      f"gate_pass={gate_pass} {METRIC}={info['score']}")

            # ── Transfer path (T9) ─────────────────────────────────────
            elif TRANSFER_ENV == "EFGHI":
                results = {}
                for cond in EXPERIMENT_CONDITIONS:
                    phases = _build_transfer_phases(
                        cond, ordered_train, ordered_verify, ordered_efghi,
                    )
                    results[cond["label"]] = _run_condition(
                        cond, phases, session_label=cond["label"],
                    )
                all_run_results.append({"run": run_idx, "results": results})
                c = _probe_acc_filtered(results.get("ceiling"),  "efghi_probe", PROBE_LEVEL_FILTER)
                l = _probe_acc_filtered(results.get("learning"), "efghi_probe", PROBE_LEVEL_FILTER)
                if c is not None and l is not None:
                    le = compute_learning_efficiency(ceiling_accuracy=c, learning_accuracy=l)
                    score = le["learning_efficiency"]
                    run_scores.append(score)
                    print(f"Run {run_idx+1}/{N_RUNS}: LE={score:.3f} (ceil={c:.3f}, learn={l:.3f})")
                run_details.append({"run": run_idx, "ceiling_acc": c, "learning_acc": l})

            # ── Interleaved schedule path (T3) ─────────────────────────
            elif SCHEDULE == "interleaved-blocks":
                results = {}
                for cond in EXPERIMENT_CONDITIONS:
                    phases = _build_interleaved_phases(cond, ordered_train, ordered_probe)
                    results[cond["label"]] = _run_condition(
                        cond, phases, session_label=cond["label"],
                    )
                all_run_results.append({"run": run_idx, "results": results})
                if METRIC == "wlc":
                    c = _wlc(results.get("ceiling"))
                    l = _wlc(results.get("learning"))
                    if c is not None and l is not None:
                        le = compute_learning_efficiency(ceiling_accuracy=c, learning_accuracy=l)
                        score = le["learning_efficiency"]
                        run_scores.append(score)
                        print(f"Run {run_idx+1}/{N_RUNS}: LE(WLC)={score:.3f} "
                              f"(WLC_ceil={c:.3f}, WLC_learn={l:.3f})")
                    run_details.append({"run": run_idx, "wlc_ceiling": c, "wlc_learning": l})
                else:
                    c = _probe_acc_filtered(results.get("ceiling"),  None, None) \
                        if False else None  # fall through to simple path
                    raise ValueError(
                        f"metric='{METRIC}' not supported with schedule='interleaved-blocks' "
                        f"(expected 'wlc')"
                    )

            # ── Simple path (T1/T2/T4/T5/T6/T7/T8) ─────────────────────
            else:
                results = {}
                for cond in EXPERIMENT_CONDITIONS:
                    phases = _build_simple_phases(cond, ordered_train, ordered_probe)
                    results[cond["label"]] = _run_condition(
                        cond, phases, session_label=cond["label"],
                    )
                all_run_results.append({"run": run_idx, "results": results})
                c = _probe_acc_filtered(results.get("ceiling"),  "main_probe", PROBE_LEVEL_FILTER)
                l = _probe_acc_filtered(results.get("learning"), "main_probe", PROBE_LEVEL_FILTER)
                if c is not None and l is not None:
                    le = compute_learning_efficiency(ceiling_accuracy=c, learning_accuracy=l)
                    score = le["learning_efficiency"]
                    run_scores.append(score)
                    print(f"Run {run_idx+1}/{N_RUNS}: LE={score:.3f} (ceil={c:.3f}, learn={l:.3f})")
                run_details.append({"run": run_idx, "ceiling_acc": c, "learning_acc": l})

    except APIFailError as _api_err:
        api_fail_reason = str(_api_err)
        print(f"API_FAIL — task nulled: {api_fail_reason}")

    # ── 13. Aggregate score ──────────────────────────────────────────
    out = {
        "task_id": TASK_ID,
        "n_runs": len(all_run_results),
        "run_scores": [round(x, 4) for x in run_scores],
        "run_details": run_details,
    }
    if run_scores:
        mean_s = float(np.mean(run_scores))
        std_s  = float(np.std(run_scores))
        out["mean_score"] = round(mean_s, 4)
        out["std_score"]  = round(std_s, 4)
        print(f"Mean {METRIC}: {mean_s:.3f} ± {std_s:.3f} ({len(run_scores)} runs)")

    if cfg.get("emit_secondaries", True):
        _emit_secondaries(all_run_results, TASK_ID, by_template=emit_per_template)

    if api_fail_reason is not None:
        out["api_fail_reason"] = api_fail_reason
        _final_score = float("nan")
    elif gate_failed_early:
        out["fail_fast"] = True
        _final_score = 0.0
    else:
        _final_score = _finalize_score(out.get("mean_score"))
    out["kaggle_final_score"] = _final_score
    print(f"\n{'═'*50}")
    print(f"  SANDBOX SCORE ({TASK_ID}): {_final_score:.4f}")
    print(f"{'═'*50}")

    global task_results
    task_results = {"all_run_results": all_run_results, "metrics": out}
    return _final_score


task_results = {}

In [ ]:
gcsg_sandbox.run(kbench.llm)

In [ ]:
# ======================================================================
# TRACE ANALYSIS  (dev)
# ======================================================================

def _show_trace_analysis(results: dict, label_prefix: str = ""):
    for cond_label, result in results.items():
        traces = result.get("all_traces", pd.DataFrame())
        if traces.empty:
            continue
        print(f"\n{'─'*60}")
        print(f"{label_prefix}{cond_label}  ({len(traces)} rows)")

        if "phase_name" in traces.columns and "goal_level" in traces.columns:
            ph = traces.groupby(["phase_name", "goal_level"]).agg(
                n=("mission_success", "size"),
                acc=("mission_success", "mean"),
            ).round(3)
            print(f"\n  By phase x goal_level:\n{ph.to_string()}")

        if "template_id" in traces.columns:
            probe_traces = traces[traces.get("stage", pd.Series(dtype=str)) != "train"]                 if "stage" in traces.columns else traces
            if not probe_traces.empty:
                t_acc = probe_traces.groupby("template_id")["mission_success"].mean().round(3)
                t_n   = probe_traces.groupby("template_id")["mission_success"].count()
                t_summary = pd.DataFrame({"acc": t_acc, "n": t_n}).sort_values("acc")
                print(f"\n  Probe acc by template_id (worst first):")
                print(t_summary.to_string())

        if "failure_label" in traces.columns:
            fails = traces[traces["failure_label"] != "NONE"]
            if not fails.empty:
                print(f"\n  Failure breakdown ({len(fails)} failures):")
                for lbl, cnt in fails["failure_label"].value_counts().items():
                    print(f"    {lbl}: {cnt}")

try:
    _show_trace_analysis(task_results.get("results", {}))
except NameError:
    print("Run the task cell first to populate task_results.")

In [ ]:
%choose gcsg_sandbox